# Grid Data Explorer
Run each cell with **Shift+Enter** to see the data and charts step by step.

## 1 — Look at the raw node data (what came from Zenodo)

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Load one raw node file — this is what the meter actually recorded
raw = pd.read_csv('data/raw/zenodo/R1S0.23T4.txt', sep=';', decimal=',')
raw.columns = ['node', 'timestamp', 'kwh']
raw['timestamp'] = pd.to_datetime(raw['timestamp'], dayfirst=True, errors='coerce')
raw = raw.dropna(subset=['timestamp']).sort_values('timestamp').reset_index(drop=True)
print(f'Rows: {len(raw)}')
raw.head(20)

Rows: 1114


,node,timestamp,kwh
0,R1S0.23T4,2019-03-01,24.608
1,R1S0.23T4,2019-03-02,24.584
2,R1S0.23T4,2019-03-03,22.720
3,R1S0.23T4,2019-03-04,23.965
4,R1S0.23T4,2019-03-05,23.555
5,R1S0.23T4,2019-03-06,24.334
6,R1S0.23T4,2019-03-07,24.960
7,R1S0.23T4,2019-03-08,24.094
8,R1S0.23T4,2019-03-09,24.140
9,R1S0.23T4,2019-03-10,22.559


## 2 — Plot one node's consumption over time

In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(raw['timestamp'], raw['kwh'], linewidth=0.7, color='steelblue')
plt.title('Node R1S0.23T4 — hourly consumption (kWh)')
plt.xlabel('Time')
plt.ylabel('kWh')
plt.tight_layout()
plt.show()

## 3 — See all 47 nodes in a table

In [ ]:
all_nodes = []
for f in sorted(Path('data/raw/zenodo').glob('*.txt')):
    df = pd.read_csv(f, sep=';', decimal=',')
    df.columns = ['node', 'timestamp', 'kwh']
    df['timestamp'] = pd.to_datetime(df['timestamp'], dayfirst=True, errors='coerce')
    df = df.dropna(subset=['timestamp'])
    all_nodes.append({
        'node': f.stem,
        'records': len(df),
        'min_kwh': round(df['kwh'].min(), 2),
        'max_kwh': round(df['kwh'].max(), 2),
        'mean_kwh': round(df['kwh'].mean(), 2),
        'first_date': df['timestamp'].min().date(),
        'last_date': df['timestamp'].max().date(),
    })

summary = pd.DataFrame(all_nodes)
print(f'Total nodes: {len(summary)}')
summary

## 4 — Which nodes have the most missing hours?

In [ ]:
import json

with open('reports/anomaly_report.json') as f:
    report = json.load(f)

missing_rows = []
for r in report:
    for a in r.get('anomalies', []):
        if a['type'] == 'missing_timestamps':
            missing_rows.append({'node': r['node'], 'missing_hours': a['count']})

missing_df = pd.DataFrame(missing_rows).sort_values('missing_hours', ascending=False)
missing_df['missing_days'] = (missing_df['missing_hours'] / 24).round(0).astype(int)
missing_df.reset_index(drop=True)

## 5 — Bar chart: missing hours per node

In [ ]:
plt.figure(figsize=(16, 5))
plt.bar(missing_df['node'], missing_df['missing_hours'], color='tomato')
plt.xticks(rotation=90, fontsize=7)
plt.title('Missing hours per node')
plt.ylabel('Missing hours')
plt.tight_layout()
plt.show()

## 6 — Find the spike: show the worst anomaly

In [ ]:
spike_node = pd.read_csv('data/raw/zenodo/R2S0.415T4.txt', sep=';', decimal=',')
spike_node.columns = ['node', 'timestamp', 'kwh']
spike_node['timestamp'] = pd.to_datetime(spike_node['timestamp'], dayfirst=True, errors='coerce')
spike_node = spike_node.dropna(subset=['timestamp']).sort_values('timestamp')

mean = spike_node['kwh'].mean()
std = spike_node['kwh'].std()
spike_node['zscore'] = (spike_node['kwh'] - mean) / std
spikes = spike_node[spike_node['zscore'] > 3]

print(f'Node mean: {mean:.1f} kWh')
print(f'Worst spike: {spike_node["kwh"].max():.1f} kWh')
print(f'Number of spikes (Z > 3): {len(spikes)}')
spikes[['timestamp','kwh','zscore']].head(10)

## 7 — Plot the spike visually

In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(spike_node['timestamp'], spike_node['kwh'], linewidth=0.7, color='steelblue', label='consumption')
plt.scatter(spikes['timestamp'], spikes['kwh'], color='red', zorder=5, label=f'spike (Z>3)', s=30)
plt.axhline(mean, color='orange', linestyle='--', label=f'mean {mean:.1f} kWh')
plt.title('Node R2S0.415T4 — spikes highlighted in red')
plt.ylabel('kWh')
plt.legend()
plt.tight_layout()
plt.show()

## 8 — All 47 nodes aggregated (hourly total)

In [ ]:
hourly = pd.read_csv('data/processed/hourly_consumption.csv', parse_dates=['timestamp'])
print(f'Rows: {len(hourly)}')
hourly.head(10)

In [ ]:
monthly = hourly.set_index('timestamp').resample('ME')['total_kwh'].sum() / 1000  # to MWh

plt.figure(figsize=(14, 4))
monthly.plot(kind='bar', color='steelblue')
plt.title('All 47 nodes — monthly total consumption (MWh)')
plt.ylabel('MWh')
plt.xticks(rotation=45, fontsize=7)
plt.tight_layout()
plt.show()

---
## DATA QUALITY TESTS
### Test 1 — Completeness: Are any timestamps missing?